In [1]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

c:\Users\bpu320145\2026_Projects\Agentic_Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    feedback: str
    is_complete: bool

In [3]:
# Fixed tool to prevent infinite loops
def simple_search(query: str) -> str:
    """Use this tool to search for factual information."""
    query = query.lower()
    if "use case" in query or "overview" in query or "feature" in query:
        return (
            "LangGraph is ideal for complex, multi-agent workflows and cyclical tasks. "
            "LCEL is designed for simple, linear data pipelines."
        )
    return "LangGraph allows for cyclic execution, unlike standard LangChain Expression Language (LCEL)."

tools = [simple_search]
worker_llm = ChatOpenAI(model="gpt-4o-mini").bind_tools(tools)

class EvaluatorDecision(BaseModel):
    feedback: str = Field(description="Critique of the assistant's answer.")
    is_complete: bool = Field(description="True if the answer fully resolves the user's prompt.")

evaluator_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(EvaluatorDecision)

In [4]:
def worker_node(state: State):
    messages = state["messages"]
    if state.get("feedback") and not state.get("is_complete"):
        correction_prompt = f"System: Your previous answer was rejected. Feedback: {state['feedback']}. Try again."
        messages = messages + [SystemMessage(content=correction_prompt)]
    
    response = worker_llm.invoke(messages)
    return {"messages": [response]}

def evaluator_node(state: State):
    last_message = state["messages"][-1].content
    user_request = state["messages"][0].content
    
    prompt = f"Original Request: {user_request}\nAssistant Answer: {last_message}\nDoes the answer resolve the request?"
    decision = evaluator_llm.invoke(prompt)
    
    return {"feedback": decision.feedback, "is_complete": decision.is_complete}

In [5]:
def worker_router(state: State) -> Literal["tools", "evaluator"]:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "evaluator"

def evaluator_router(state: State) -> Literal["worker", "END"]:
    if state.get("is_complete"):
        return "END"
    return "worker"

In [6]:
# 1. Initialize MemorySaver
memory = MemorySaver()

builder = StateGraph(State)
builder.add_node("worker", worker_node)
builder.add_node("tools", ToolNode(tools=tools))
builder.add_node("evaluator", evaluator_node)

builder.add_edge(START, "worker")
builder.add_conditional_edges("worker", worker_router)
builder.add_edge("tools", "worker")
builder.add_conditional_edges("evaluator", evaluator_router, {"worker": "worker", "END": END})

# 2. Compile the graph with the checkpointer
graph = builder.compile(checkpointer=memory)

# 3. Execution configuration specifying the thread ID
config = {"configurable": {"thread_id": "session_1"}}

In [7]:
print("--- Turn 1 ---")
inputs = {"messages": [HumanMessage(content="What does LangGraph do differently than LCEL?")]}
for chunk in graph.stream(inputs, config=config, stream_mode="updates"):
    if 'worker' in chunk and not chunk['worker']['messages'][-1].tool_calls:
        print("Assistant:", chunk['worker']['messages'][-1].content)

--- Turn 1 ---
Assistant: LangGraph and LCEL differ primarily in their design and intended use cases.

- **LangGraph** is tailored for complex, multi-agent workflows and cyclical tasks, making it suitable for scenarios where multiple processes need to interact in intricate ways.

- **LCEL**, on the other hand, is aimed at simple, linear data pipelines, which are more straightforward and generally follow a one-way process.

In summary, if your project involves intricate interactions and requires handling multiple agents, LangGraph is more appropriate, while LCEL works best for simpler, linear workflows.


c:\Users\bpu320145\2026_Projects\Agentic_Chatbot\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvaluatorDecision(feedbac...er.', is_complete=False), input_type=EvaluatorDecision])
  return self.__pydantic_serializer__.to_python(


Assistant: LangGraph and LCEL serve different purposes and excel in distinct areas.

### LangGraph:
- **Functionality**: LangGraph is designed for handling complex, multi-agent workflows and cyclical tasks. It allows for iterative processes where outputs can circle back as inputs through multiple agents or components.
- **Use Cases**:
  - **Project Management**: Managing tasks that require input from various departments and may loop back for revisions.
  - **Data Analysis**: Iterating through data processing where results inform and refine the analysis multiple times.
- **Example Scenario**: In a marketing automation platform where different teams (content, design, analytics) must collaborate on campaigns that are evaluated and adjusted over time.

### LCEL (LangChain Expression Language):
- **Functionality**: LCEL is focused on simpler, linear data pipelines. It is designed for straightforward workflows where data moves in a single direction without the need for iterative feedback loo

c:\Users\bpu320145\2026_Projects\Agentic_Chatbot\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvaluatorDecision(feedbac...ers.', is_complete=True), input_type=EvaluatorDecision])
  return self.__pydantic_serializer__.to_python(


In [8]:
print("\n--- Turn 2 (Testing Memory) ---")
# The LLM should know you are referring to LangGraph and LCEL without explicit mention
inputs_2 = {"messages": [HumanMessage(content="Which one of those two is better for a coding assistant?")]}
for chunk in graph.stream(inputs_2, config=config, stream_mode="updates"):
    if 'worker' in chunk and not chunk['worker']['messages'][-1].tool_calls:
        print("Assistant:", chunk['worker']['messages'][-1].content)


--- Turn 2 (Testing Memory) ---
Assistant: When evaluating which tool—LangGraph or LCEL—is better suited for a coding assistant, consider the nature of the tasks involved:

### LangGraph:
- **Strengths**: LangGraph is designed for handling complex, multi-agent workflows and cyclical tasks. This can be beneficial for a coding assistant that needs to:
  - Interact with multiple components or tools (like code editors, compilers, and testing frameworks).
  - Provide iterative feedback, allowing for refinement of code through debugging and testing loops.
  - Manage interactions between various programming languages or frameworks in a dynamic way.

- **Use Case in Coding**: A coding assistant implemented with LangGraph could allow developers to receive suggestions that evolve based on their coding style, correct syntax, and integrated testing results, leading to a more interactive and responsive coding experience.

### LCEL:
- **Strengths**: LCEL is built for simpler, linear data pipelines,

c:\Users\bpu320145\2026_Projects\Agentic_Chatbot\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvaluatorDecision(feedbac...ew.', is_complete=False), input_type=EvaluatorDecision])
  return self.__pydantic_serializer__.to_python(


Assistant: When considering a coding assistant, it’s important to compare LangGraph and LCEL not only in terms of their strengths but also regarding their weaknesses and how they fit into typical coding tasks.

### LangGraph:
- **Strengths**:
  - Offers support for complex, multi-agent workflows, making it ideal for interactive coding environments where multiple tools or processes need to work together.
  - Facilitates iterative feedback loops, allowing the assistant to refine its suggestions based on previous interactions, which is crucial for debugging and code optimization.
- **Potential Weaknesses**:
  - The complexity of setting up and managing workflows may introduce overhead, potentially overwhelming new users who require straightforward coding assistance.
  - It might require more resources and time to implement due to its complex nature.

### LCEL:
- **Strengths**:
  - Simplicity in design allows for quick implementation, making it suitable for straightforward coding tasks suc

c:\Users\bpu320145\2026_Projects\Agentic_Chatbot\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvaluatorDecision(feedbac...es.", is_complete=False), input_type=EvaluatorDecision])
  return self.__pydantic_serializer__.to_python(


Assistant: When evaluating LangGraph and LCEL for a coding assistant, it's essential to highlight their unique functionalities and how they apply specifically to coding tasks.

### Differences Between LangGraph and LCEL for a Coding Assistant

1. **Workflow Complexity**:
   - **LangGraph**: 
     - Specifically designed for complex, multi-agent workflows. This means it can manage interactions among various tools, libraries, and components during programming tasks. 
     - Example: A coding assistant using LangGraph could facilitate interactions between an IDE (integrated development environment), version control systems, and testing frameworks. For instance, it could pull the latest changes from version control, run unit tests, and suggest code improvements based on test results—all in one integrated environment.
   - **LCEL**:
     - Focused on simple, linear data pipelines, it is better suited for straightforward tasks without complex interdependencies.
     - Example: An assistant b

c:\Users\bpu320145\2026_Projects\Agentic_Chatbot\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvaluatorDecision(feedbac...ely.", is_complete=True), input_type=EvaluatorDecision])
  return self.__pydantic_serializer__.to_python(
